# 01: Data Cleaning & Transformation
***
**Author:** Zain Zabidi
**Date:** February 2026

**Objective:** To clean raw Census CSVs, filter for Western Australian data, and apply midpoint mappings to create a consolidated numerical dataset for analysis.

## 1. Environment Setup & Data Loading
First we import the raw Census CSVs and the midpoint dictionaries.

**Datasets to be processed:**
* **G01 & G02:** Baseline population characteristics and official ABS medians for validation.
* **G33, G38 , G40:** Categorical data requiring midpoint mapping (Income, Mortgage, Rent).
* **G46, G54:** Count-based data (Labour Force, Industry).

### 1.1 Import Libraries and Define Paths
Using **Pandas** for transformation and **OS/JSON** for file management and dictionary retrieval.

In [2]:
import pandas as pd
import os
import json

# Set directory paths for raw data, dictionaries, and processed output
RAW_DATA_PATH = '../data/raw/'
DICT_PATH = '../data/dictionary/'
PROCESSED_DATA_PATH = '../data/processed/'

# Ensure processed directory exists
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

# List all required ABS Census tables for the project
G_FILES = [
    'G01', 'G02', 'G33', 'G38', 'G40', 'G41', 'G46A', 'G46B', 'G54A', 'G54B'
]

# Mapping dicts created in notebook 00 to respective data types
DICT_FILES = {
    'income': 'income_midpoints.json',
    'mortgage': 'mortgage_midpoints.json',
    'rent': 'rent_midpoints.json'
}

### 1.2 Data Ingestion
We load the CSV files into a dictionary of DataFrames.These datasets were acquired already pre-filtered for Western Australia.

In [3]:
# Dict to store the filtered DataFrames
dfs = {}

for g_file in G_FILES:
    # Construct file path
    file_name = f'2021Census_{g_file}_WA_SAL.csv'
    full_path = os.path.join(RAW_DATA_PATH, file_name)

    # Load dataset
    dfs[g_file] = pd.read_csv(full_path)

# Verify number of rows in samples
for name, df in dfs.items():
    print(f"{name}: {df.shape[0]} records")

G01: 1701 records
G02: 1701 records
G33: 1701 records
G38: 1701 records
G40: 1701 records
G41: 1701 records
G46A: 1701 records
G46B: 1701 records
G54A: 1701 records
G54B: 1701 records


### 1.3 Load Mapping Dictionaries
We load the midpoint dictionaries previously exported to JSON. These will be used to transform categorical range data into continuous numerical values.

In [6]:
# Dict to store loaded midpoint mappings
midpoint_dicts = {}

for key, filename in DICT_FILES.items():
    dict_full_path = os.path.join(DICT_PATH, filename)

    with open(dict_full_path, 'r') as f:
        midpoint_dicts[key] = json.load(f)

# Check data loaded correctly
for key, mapping in midpoint_dicts.items():
    sample = dict(list(mapping.items())[:3])
    print(f"{key.capitalize()} mapping sample: {sample}")

Income mapping sample: {'Negative_Nil_income_Tot': 0, 'HI_1_149_Tot': 75.0, 'HI_150_299_Tot': 224.5}
Mortgage mapping sample: {'M_0_299_Tot': 149.5, 'M_300_449_Tot': 374.5, 'M_450_599_Tot': 524.5}
Rent mapping sample: {'R_1_74_Tot': 37.5, 'R_75_99_Tot': 87.0, 'R_100_149_Tot': 124.5}


## Section 2: Data Transformation
In this section, we apply the midpoint mappings to the Census tables. We will process each major variable (Rent, Mortgage, Income) individually to ensure accuracy before merging the features into a single dataset.

### 2.1 Process Weekly Rent (G40)
We apply the rent midpoint dictionary to the G40 table. This allows us to estimate the total weekly rent expenditure per suburb by multiplying household counts by bracket midpoints.

**Formula:**
$$Total\_Rent\_Weekly = \sum (Household\_Count_{range} \times Midpoint_{range})$$

*Where:*
* ***Household_Count:** Number of households reported in a specific rent range (e.g., $150-$199).*
* ***Midpoint:** The numerical representative value for that range (e.g., $174.5).*
* ***Total:** The sum of all rent paid by all reporting households in the suburb combined.*

In [9]:
# Identify columns in G40 that match to our rent midpoint keys
rent_cols = list(midpoint_dicts['rent'].keys())

# Create copy of G40 dataframe to alter
rent_df = dfs['G40'].copy()

# Calculate estimated total rent for each bracket
# Multiply household count by midpoint value
for col in rent_cols:
    midpoint = midpoint_dicts['rent'][col]
    rent_df[f'{col}_val'] = rent_df[col] * midpoint

# Sum calculated values for single 'total_rent_weekly' column
val_cols = [f'{col}_val' for col in rent_cols]
rent_df['total_rent_weekly'] = rent_df[val_cols].sum(axis=1)

# Display check for first few results
display(rent_df[['SAL_CODE_2021', 'total_rent_weekly']].head())

,SAL_CODE_2021,total_rent_weekly
0,SAL50001,0.0
1,SAL50002,38475.5
2,SAL50003,998.0
3,SAL50004,0.0
4,SAL50005,0.0


**Observation:**
The `total_rent_weekly` value represents the estimated aggregate rent for the entire suburb.
* For example, **SAL50002** shows ~$38,475, which is the sum of all rent paid by all households that fell within our defined price ranges.
* Suburbs with **0.0** (like SAL50001) indicate areas with no reported rental households in those specific Census categories (likely industrial areas, parks, or zero-population zones).

This aggregate figure serves as our **numerator** for calculating the mean weekly rent in the next step.

### 2.2 Calculate Renting Households
To calculate a meaningful average, we need to know how many households contributed to the total rent expenditure. We sum the counts from the same price ranges used in the previous step.

In [10]:
# Sum counts of households in relevant rent price ranges
rent_df['renting_households_count'] = rent_df[rent_cols].sum(axis=1)

# Calculate Average Weekly Rent per Suburb
# Avoid dividing by zero for no-renter suburbs with a small check
rent_df['avg_weekly_rent'] = (
    rent_df['total_rent_weekly'] / rent_df['renting_households_count']
).fillna(0)

# Display check
display(rent_df[['SAL_CODE_2021', 'total_rent_weekly', 'renting_households_count', 'avg_weekly_rent']].head())

,SAL_CODE_2021,total_rent_weekly,renting_households_count,avg_weekly_rent
0,SAL50001,0.0,0,0.000000
1,SAL50002,38475.5,104,369.956731
2,SAL50003,998.0,4,249.500000
3,SAL50004,0.0,0,0.000000
4,SAL50005,0.0,0,0.000000
